# 31c — Stage-C learning-rate stability ablation

Paired three-seed comparison of the exact Notebook 31 model at Stage-C learning rates `0.001` and `0.0003`. The challenger reuses the baseline's identical Stage-A encoder and Stage-B expert bank for each seed, so only Stage C is retrained. All current task, resource, training-history, and A/B/C latent reports are produced.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDIES = [
    ('baseline_lr1e3', 'config/compact_soft_moe_3seed.yaml', 'nfv3_4way_moe_soft_comparable_repro_v1'),
    ('stagec_lr3e4', 'config/compact_soft_moe_stage_c_lr3e4_3seed.yaml', 'nfv3_4way_moe_soft_stagec_lr3e4_v1'),
]
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS_PER_STUDY = 3
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO
url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None
os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR
os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Dry-run and resumable execution

The baseline is completed first if necessary. The `0.0003` challenger then loads that seed's Stage-A and Stage-B checkpoints and trains only Stage C. A changed protocol requires a new prefix.

In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before executing the study.')
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py', 'tests/test_training_observability.py'], check=True)
for condition, study_config, prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'
    summary_dir.mkdir(parents=True, exist_ok=True)
    print(f'=== {condition}: expected parameters={study["expected_total_parameters"]:,} ===')
    command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', study_config, '--prefix', prefix]
    run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31c-{condition}-dry-run')
    if EXECUTE:
        launch = [*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS_PER_STUDY)]
        run_streaming_logged(launch, str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31c-{condition}-training')
if not EXECUTE:
    print('NO TRAINING WAS STARTED. Review the graph, set EXECUTE=True, and rerun this cell.')

## Paired validation and locked-test comparison

In [ ]:
import pandas as pd
from IPython.display import Image, display
summaries = {condition: Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' for condition, _, prefix in STUDIES}
complete = {name: (path / 'Final_Study_Manifest.json').is_file() for name, path in summaries.items()}
print('Completion:', complete)
if all(complete.values()):
    validation = []
    test = []
    resources = []
    for name, path in summaries.items():
        frame = pd.read_csv(path / 'Three_Seed_Guardrails.csv'); frame.insert(0, 'condition', name); validation.append(frame)
        frame = pd.read_csv(path / 'Three_Seed_Overall.csv'); frame.insert(0, 'condition', name); test.append(frame)
        frame = pd.read_csv(path / 'Three_Seed_Resource_Accounting.csv'); frame.insert(0, 'condition', name); resources.append(frame)
    validation = pd.concat(validation, ignore_index=True)
    test = pd.concat(test, ignore_index=True)
    print('=== Primary validation comparison ==='); display(validation[['condition', 'seed', 'validation_macro_f1', 'worst_dataset_macro_f1', 'rare_recall_mean']])
    pivot = validation.pivot(index='seed', columns='condition', values='validation_macro_f1')
    pivot['delta_lr3e4_minus_baseline'] = pivot['stagec_lr3e4'] - pivot['baseline_lr1e3']; display(pivot)
    metrics = [c for c in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
    print('=== Locked-test descriptive mean and SD ==='); display(test.groupby('condition')[metrics].agg(['mean', 'std']))
    print('=== Resource accounting ==='); display(pd.concat(resources, ignore_index=True))
else:
    print('Comparison appears after both three-seed manifests are complete.')

## Training stability and latent-space comparison

In [ ]:
if all(complete.values()):
    histories = []
    latent = []
    for name, path in summaries.items():
        frame = pd.read_csv(path / 'Training_History_3Seed.csv'); frame.insert(0, 'condition', name); histories.append(frame)
        frame = pd.read_csv(path / 'Latent_Snapshot_Summary_3Seed.csv'); frame.insert(0, 'condition', name); latent.append(frame)
    stage_c = pd.concat(histories, ignore_index=True).query("stage == 'C'")
    display(stage_c[['condition', 'seed', 'epoch', 'learning_rate', 'train_total_loss', 'val_macro_f1', 'best_val_macro_f1', 'patience_left']])
    display(pd.concat(latent, ignore_index=True))
    for name, path in summaries.items():
        print('===', name, 'training figures ===')
        for figure in sorted((path / 'training_figures').glob('*.png')):
            display(Image(filename=str(figure)))